# Canonical Kaggle DeepLabV3+ Reproduction

This notebook is the clean Run All workflow for the sealed semantic baseline. It delegates behavior to repository modules and contains no source-edit recovery patches. Expert evaluation occurs only after the development-selected checkpoint is frozen.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/mandevautospa/AI4Mars.git'
REPOSITORY_REF = 'main'  # Replace with the reviewed immutable commit/tag when reproducing a specific run.
WORKDIR = Path('/kaggle/working/AI4Mars')
DATASET_SLUG = 'REPLACE_WITH_AI4MARS_DATASET_SLUG'
DATASET_ROOT = Path('/kaggle/input') / DATASET_SLUG
OUTPUT_ROOT = Path('/kaggle/working/ai4mars-paper-reproduction')

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF, REPOSITORY_URL, str(WORKDIR)], check=True)

In [ ]:
# The repository now exists, so its Kaggle-specific dependency list is available.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], cwd=WORKDIR, check=True)

In [ ]:
CONFIG = 'configs/reproduction/paper_deeplabv3plus_kaggle_p100.yaml'
RUN_ID = 'paper-deeplabv3plus-kaggle-p100'
RESUME_CHECKPOINT = None  # Set to an attached prior run's last.pth, or leave None for a fresh 40-epoch run.

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Attach the AI4Mars dataset and set DATASET_SLUG. Missing: {DATASET_ROOT}')
subprocess.run(['nvidia-smi'], check=False)
subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.version.cuda)'], check=True)

In [ ]:
# Metadata validation is deterministic and does not read image pixels.
subprocess.run([
    sys.executable, '-m', 'src.train', '--config', CONFIG,
    '--dataset-root', str(DATASET_ROOT), '--output-root', str(OUTPUT_ROOT),
    '--validate-only', '--validation-level', 'metadata',
], cwd=WORKDIR, check=True)

In [ ]:
# Full 40-epoch reproduction or source-compatible resume. No notebook monkey-patching is needed.
train_command = [
    sys.executable, '-m', 'src.train', '--config', CONFIG,
    '--dataset-root', str(DATASET_ROOT), '--output-root', str(OUTPUT_ROOT), '--run-id', RUN_ID,
]
if RESUME_CHECKPOINT:
    train_command.extend(['--resume-checkpoint', str(RESUME_CHECKPOINT)])
subprocess.run(train_command, cwd=WORKDIR, check=True)

In [ ]:
# Freeze and identify the development-selected checkpoint before sealed expert evaluation.
import hashlib

CHECKPOINT = OUTPUT_ROOT / 'runs' / RUN_ID / 'checkpoints' / 'best_val_miou.pth'
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f'Development-selected checkpoint missing: {CHECKPOINT}')
digest = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
print('Frozen checkpoint:', CHECKPOINT)
print('SHA-256:', digest)

In [ ]:
# Sealed final evaluation. This is not a training, resume, or checkpoint-selection input.
EXPERT_EVAL_RUN_ID = f'{RUN_ID}-expert-evaluation'
subprocess.run([
    sys.executable, '-m', 'src.paper_evaluate', '--config', CONFIG,
    '--checkpoint', str(CHECKPOINT), '--dataset-root', str(DATASET_ROOT),
    '--output-root', str(OUTPUT_ROOT), '--run-id', EXPERT_EVAL_RUN_ID,
    '--splits', 'expert_min1', 'expert_min2', 'expert_min3',
], cwd=WORKDIR, check=True)

In [ ]:
# Regenerate publication-resolution confusion artifacts from saved evaluation data only.
EVALUATION_ARTIFACT = OUTPUT_ROOT / 'runs' / EXPERT_EVAL_RUN_ID / 'artifacts' / 'expert_evaluation.json'
PLOT_DIR = OUTPUT_ROOT / 'runs' / EXPERT_EVAL_RUN_ID / 'artifacts' / 'error_analysis'
subprocess.run([
    sys.executable, '-m', 'src.paper_error_analysis',
    '--evaluation-artifact', str(EVALUATION_ARTIFACT), '--output-dir', str(PLOT_DIR),
], cwd=WORKDIR, check=True)